In [1]:
import os
import cantera as ct
import numpy as np
import pandas as pd
import ess
import glob
import sys

import yaml
import logging

import arviz
import zeus

import matplotlib.pyplot as plt
%matplotlib inline

/scratch/harris.se/guassian_scratch/mk3_env/lib/python3.14/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


In [2]:
logging.basicConfig()
logging.getLogger().setLevel(logging.INFO)

In [3]:
working_dir = '/scratch/harris.se/guassian_scratch/mk3_runs/multi_20260210/'
correlated = True
if correlated:
    label = 'corr'
else:
    label = 'uncorr'
example_run_dir = os.path.join(working_dir, f'cpox_{label}', f'cpox_{label}_0000')

In [4]:
sim_info_yaml = os.path.join(example_run_dir, 'sim_info.yaml')
with open(sim_info_yaml) as f:
    sim_info = yaml.safe_load(f)

In [21]:
chain_0s = sorted(glob.glob(os.path.join(working_dir, f'cpox_{label}', '*', 'results', 'chain_0.npy')))
chain_1s = sorted(glob.glob(os.path.join(working_dir, f'cpox_{label}', '*', 'results', 'chain_1.npy')))

logp_0s = sorted(glob.glob(os.path.join(working_dir, f'cpox_{label}', '*', 'results', 'logPs_0.npy')))
logp_1s = sorted(glob.glob(os.path.join(working_dir, f'cpox_{label}', '*', 'results', 'logPs_1.npy')))

In [22]:
# concatenate all chains and logPs

N_burn_in = 50


chain0 = np.load(chain_0s[0])[N_burn_in:, :, :]
logP0 = np.load(logp_0s[0])[N_burn_in:]


chain1 = np.load(chain_1s[0])[N_burn_in:, :, :]
logP1 = np.load(logp_1s[0])[N_burn_in:]

chain = np.concatenate((chain0, chain1))
logP = np.concatenate((logP0, logP1))

for i in range(1, len(chain_0s)):
    chain0 = np.load(chain_0s[i])[N_burn_in:, :, :]
    logP0 = np.load(logp_0s[i])[N_burn_in:]

    print(chain0.shape)
    
    
    chain1 = np.load(chain_1s[i])[N_burn_in:, :, :]
    logP1 = np.load(logp_1s[i])[N_burn_in:]
    
    chain = np.concatenate((chain, chain0))
    chain = np.concatenate((chain, chain1))
    logP = np.concatenate((logP, logP0))
    logP = np.concatenate((logP, logP1))

assert chain.shape[0] == logP.shape[0]

(2171, 22, 11)
(2027, 22, 11)
(2228, 22, 11)
(1932, 22, 11)
(1914, 22, 11)
(1296, 22, 11)
(2094, 22, 11)


In [7]:
# need to throw out ~150 samples first

In [24]:
25000 / 8 / 22

142.04545454545453

In [25]:
25000 / 22

1136.3636363636363

In [9]:
chain.shape[0]

28174

In [8]:
26612 * 16

425792

In [31]:
19427*16

310832

In [12]:
1562*16

24992

In [10]:
28174 - 26612

1562

In [30]:
# next do geweke drift
flat_chain = ess.flatten_chain(chain0)
for i in range(chain0.shape[2]):

    a = flat_chain[:len(flat_chain) // 4, i]
    b = flat_chain[-len(flat_chain) // 4:, i]
    geweke_z = (a.mean() - b.mean()) / (np.var(a) + np.var(b))**0.5
    # assert geweke_z < 1.0
    print(geweke_z)

-0.8163167051955919
-0.16936680813396418
0.6732242874342214
0.6258150242236886
1.4373035178714793
-0.21271362316870215
-0.16276104224215143
-0.19899271435006918
-0.20674709442240216
0.22023778155865417
0.5603436730987849


In [33]:
flat_chain.shape

(28600, 11)

In [8]:
min_chain_len = np.inf
for i in range(len(chain_0s)):
    chain0 = np.load(chain_0s[i])
    chain1 = np.load(chain_1s[i])
    if chain0.shape[0] < min_chain_len:
        min_chain_len = chain0.shape[0]
    if chain1.shape[0] < min_chain_len:
        min_chain_len = chain1.shape[0]


chains = []
for i in range(len(chain_0s)):
    chain0 = np.load(chain_0s[i])
    chain1 = np.load(chain_1s[i])
    chains.append(ess.flatten_chain(chain0[:min_chain_len, :, :]))
    chains.append(ess.flatten_chain(chain1[:min_chain_len, :, :]))
chains = np.asarray(chains)

In [9]:
rhat = arviz.rhat(arviz.convert_to_dataset(chains)).x.data

In [10]:
rhat

array([1.06201533, 1.03514213, 1.05412586, 1.09078358, 1.05031285,
       1.0404652 , 1.07086059, 1.06512392, 1.06894298, 1.03782444,
       1.07139812])

In [23]:
rhat

array([1.06507255, 1.03415689, 1.05459398, 1.09148214, 1.05217232,
       1.04185681, 1.07122809, 1.0687715 , 1.07066156, 1.03769475,
       1.07201469])

In [20]:
# chain0

In [11]:
tau = zeus.autocorr.AutoCorrTime(chain0)

# tau = zeus.autocorr.AutoCorrTime(chain[:, 0:1, :])
# # estimate autocorrelation time at increasing # samples that is uniformly spaced on log scale
# window_indices = np.logspace(0, np.log10(N_samples), N_taus).astype(int)

# taus = np.zeros((N_taus, N_parameters)) 
# for i, w in enumerate(window_indices):
#     taus[i, :] = zeus.autocorr.AutoCorrTime(chain[:w, :, :])


In [12]:
tau

array([ 953.80580041,  881.5206077 , 2778.38128503, 1424.05570339,
       3050.84711365, 1005.65866722, 2317.74411064, 1195.28291784,
       2589.18235348, 1066.35907294, 1152.07473836])

In [41]:
tau = zeus.autocorr.AutoCorrTime(chain, method='dfm')
print(tau)

[1025.87334664 1014.23522702 1027.0063635   964.80835905 1193.02551428
 1090.26635447 1092.71402242  994.2681501  1022.17643353  933.75638299
 1097.18684807]


In [37]:
tau = zeus.autocorr.AutoCorrTime(chain, method='gw')
print(tau)

[ 677.19739189 1989.25819836 1075.96072714  693.4194072   565.87829924
  645.84921303  561.60345302 1579.11185989  609.99130336  693.36617186
 1039.92790464]


In [ ]:
tau[0]

In [ ]:
chain[:,:,i].T

In [18]:
# # flat laod chain and logP
# chain0 = ess.flatten_chain(np.load(chain_0s[0]))
# logP0 = ess.flatten_logP(np.load(logp_0s[0]))

# chain1 = ess.flatten_chain(np.load(chain_1s[0]))
# logP1 = ess.flatten_logP(np.load(logp_0s[0]))

# chain = np.concatenate((chain0, chain1))
# logP = np.concatenate((logP0, logP1))

# for i in range(1, len(chain_0s)):
#     chain0 = ess.flatten_chain(np.load(chain_0s[i]))_autocorr_func_1d(y.reshape((-1), order='C'))
#     logP0 = ess.flatten_logP(np.load(logp_0s[i]))
#     chain1 = ess.flatten_chain(np.load(chain_1s[i]))
#     logP1 = ess.flatten_logP(np.load(logp_1s[i]))
    
#     chain = np.concatenate((chain, chain0))
#     chain = np.concatenate((chain, chain1))
#     logP = np.concatenate((logP, logP0))
#     logP = np.concatenate((logP, logP1))

# assert chain.shape[0] == logP.shape[0]

In [ ]:
sim_info

In [26]:
# chain = np.load(chain_0s[2])_autocorr_func_1d(y.reshape((-1), order='C'))
parameter_names = sim_info['parameter_names']
outdir = './'

# chain should be N_samples x N_walkers x N_parameters, or NxWxP
N_taus = 15
N_samples, N_walkers, N_parameters = chain.shape
if parameter_names is not None:
    assert len(parameter_names) == N_parameters

# estimate autocorrelation time at increasing # samples that is un_autocorr_func_1d(y.reshape((-1), order='C'))iformly spaced on log scale
window_indices = np.logspace(0, np.log10(N_samples), N_taus).astype(int)

taus = np.zeros((N_taus, N_parameters)) 
for i, w in enumerate(window_indices):
    taus[i, :] = zeus.autocorr.AutoCorrTime(chain[:w, :, :])

# make individual autocorrelation plots
for i in range(N_parameters):
    plt.figure()
    plt.loglog(window_indices, taus[:, i], marker='o', label=r'Estimated $\tau$')
    plt.plot(window_indices, window_indices / 50.0, linestyle='dashed', color='black', label=r'$\tau$=N/50')

    param_name = f'p{i}'
    if parameter_names is not None:
        param_name = parameter_names[i]
    outfile = os.path.join(outdir, f'autocorr_{param_name}.png')
    plt.xlabel('N Samples')
    plt.ylabel(param_name + ' -- ' + r'Estimated $\tau$')
    plt.legend()
    plt.savefig(outfile, bbox_inches='tight')
    plt.close()

# Make a combined autocorrelation plot
plt.figure()
for i in range(N_parameters):
    param_name = f'p{i}'
    if parameter_names is not None:
        param_name = parameter_names[i]
    plt.loglog(window_indices, taus[:, i], marker='o', label=param_name)

plt.plot(window_indices, window_indices / 50.0, linestyle='dashed', color='black', label=r'$\tau$=N/50')
plt.xlabel('N Samples')
plt.ylabel(r'Estimated $\tau$')
plt.legend()
outfile = os.path.join(outdir, f'combined_autocorr.png')
plt.savefig(outfile, bbox_inches='tight')
plt.close()

In [14]:
np.log(chain.shape

(30787, 22, 11)

In [21]:
window_indices * N_walkers

array([    22,     44,     88,    198,    396,    858,   1804,   3762,
         7876,  16434,  34298,  71566, 149292, 311432, 649638])

In [ ]:
result = zeus.autocorr.AutoCorrTime(chain[:w, :, :])

In [ ]:
chain[:w, :, :].shape

In [ ]:
np.log10(211464)